## Model Interpreter Example Notebook for a Non-Standard Model

### Setup

Will install a package with a rarer model type. Completed separately from the development as this is only used as an example and is not used in any other package file

In [1]:
pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.1/236.1 kB 23.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import sys
import numpy as np
import pandas as pd

sys.path.append("../")

import model_interpreter
from model_interpreter.interpreter import ModelInterpreter

/home/vscode/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_interpreter.__version__

'1.0.0'

### Prepare Data

The non-standard model is the BaggingClassifier

In [4]:
X, y = make_classification(
    n_samples=1000,
    n_features=4,
    n_informative=2,
    n_redundant=0,
    random_state=0,
    shuffle=False,
)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

In [6]:
feature_names = ["feature1", "feature2", "feature3", "feature4"]

### Train model

Instantiate the base classifier:

In [7]:
base_clf = DecisionTreeClassifier()

Instantiate BalancedBaggingClassifier

In [8]:
model = BalancedBaggingClassifier(
    estimator=base_clf,
    sampling_strategy="auto",
    replacement=False,
    random_state=42,
)

Fit the model:

In [9]:
model.fit(X_train, y_train)

BalancedBaggingClassifier(estimator=DecisionTreeClassifier(), random_state=42)

### Build single model contribution object

A model interpreter object is created by specifying the feature names in the same order as used for building the model.

In [10]:
single_model_contribution = ModelInterpreter(feature_names)

### Define single row of data to get contributions for

In [11]:
single_row = X_test[0].reshape(1, -1)

### Build Explainer

The fit method needs to be called on the model to build the explainer. As the model is non-standard, the data also must be passed.

Additionally, n_samples defaults to 50 as the sample size to build the explainer from but can be increased/decreased

In [12]:
single_model_contribution.fit(model, X_train=X_train, is_classification=True)

### Use the transform method to get feature contributions

The transform method is called on the single row to create the class output, which by default returns a sorted list of dictionaries of features with descending absolute contribution to the prediction in the format:

`[{feature_name: feature_contribution}, ... ]`

In [13]:
contribution_list = single_model_contribution.transform(
    single_row, return_feature_values=False
)

print(contribution_list)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  9.17it/s]

[{'Name': 'feature2', 'Value': -0.4400000000000002}, {'Name': 'feature1', 'Value': 0.0146666666666667}, {'Name': 'feature3', 'Value': -0.009}, {'Name': 'feature4', 'Value': 0.0083333333333334}]


return_feature_values can also be set to True so the return is:

`[{feature_name: (feature_value, feature_contribution)}, ... ]`

In [14]:
contribution_list = single_model_contribution.transform(
    single_row, return_feature_values=True, return_type="dicts"
)

print(contribution_list)

100%|██████████| 1/1 [00:00<00:00, 66.29it/s]

[{'feature2': (np.float64(-0.9405037129106437), -0.4400000000000002)}, {'feature1': (np.float64(1.37105505746784), 0.0146666666666667)}, {'feature3': (np.float64(-0.48948241875503073), -0.009)}, {'feature4': (np.float64(0.7635414664859205), 0.0083333333333334)}]


this however is not implemented when return_type is the default value of "name_value_dicts"

In [15]:
contribution_list = single_model_contribution.transform(
    single_row, return_feature_values=True
)

print(contribution_list)

100%|██████████| 1/1 [00:00<00:00, 90.74it/s]

[{'Name': 'feature2', 'Value': -0.4400000000000002, 'FeatureValue': np.float64(-0.9405037129106437)}, {'Name': 'feature1', 'Value': 0.0146666666666667, 'FeatureValue': np.float64(1.37105505746784)}, {'Name': 'feature3', 'Value': -0.009, 'FeatureValue': np.float64(-0.48948241875503073)}, {'Name': 'feature4', 'Value': 0.0083333333333334, 'FeatureValue': np.float64(0.7635414664859205)}]


`feature_value_row` can be used to provide a different row for feature display values while keeping feature importance values based on original `single_row`.


In [16]:
# Example using a numpy array as the display row
display_row = np.array([[10.0, 20.0, 30.0, 40.0]])

contribution_with_feature_values = single_model_contribution.transform(
    single_row,
    return_feature_values=True,
    return_type="name_value_dicts",
    feature_value_row=display_row,
)

print(contribution_with_feature_values)

# Example using a pandas DataFrame as the display row
feature_value_frame = pd.DataFrame(
    [[10.0, 20.0, 30.0, 40.0]], columns=feature_names
)

contribution_with_feature_values_df = single_model_contribution.transform(
    single_row,
    return_feature_values=True,
    return_type="name_value_dicts",
    feature_value_row=feature_value_frame,
)

print(contribution_with_feature_values_df)

100%|██████████| 1/1 [00:00<00:00, 81.87it/s]


[{'Name': 'feature2', 'Value': -0.4400000000000002, 'FeatureValue': np.float64(20.0)}, {'Name': 'feature1', 'Value': 0.0146666666666667, 'FeatureValue': np.float64(10.0)}, {'Name': 'feature3', 'Value': -0.009, 'FeatureValue': np.float64(30.0)}, {'Name': 'feature4', 'Value': 0.0083333333333334, 'FeatureValue': np.float64(40.0)}]


100%|██████████| 1/1 [00:00<00:00, 69.30it/s]

[{'Name': 'feature2', 'Value': -0.4400000000000002, 'FeatureValue': np.float64(20.0)}, {'Name': 'feature1', 'Value': 0.0146666666666667, 'FeatureValue': np.float64(10.0)}, {'Name': 'feature3', 'Value': -0.009, 'FeatureValue': np.float64(30.0)}, {'Name': 'feature4', 'Value': 0.0083333333333334, 'FeatureValue': np.float64(40.0)}]
